# COMP 4330SEF P04 Group 5: LightRAG vs Naive RAG (Experiment 1)

This notebook runs the whole pipeline: **setup → data → naive RAG baseline → LightRAG index → answering → evaluation**.

**Before running**
1. Choose a backend in the config cell:
   * `BACKEND = "openai"` (as in the proposal): GPT-4o-mini plus `text-embedding-3-small`. A **CPU runtime** is enough. Add your key under Colab's 🔑 **Secrets** panel with the name `OPENAI_API_KEY` and turn on notebook access.
   * `BACKEND = "ollama"`: free local `qwen2.5:14b` plus `bge-m3`. It needs a **T4 GPU runtime** and is much slower (about 3–6 hours to index).
2. Run all cells with **Runtime → Run all**.

Progress is saved to Google Drive (`MyDrive/COMP4330_LightRAG`). If Colab disconnects, run all again. Indexing, answering and scoring continue from where they stopped.

In [ ]:
%pip install -q lightrag-hku ragas "langchain-community<0.4" openai tiktoken huggingface_hub pandas

## 1. Config

In [ ]:
import os, sys, json, time, math, re, string, hashlib, collections, subprocess
from pathlib import Path
import numpy as np

IN_COLAB = "google.colab" in sys.modules
BACKEND = "openai"            # "openai" (proposal: GPT-4o-mini API) or "ollama" (free, needs T4 GPU)

TARGET_TOKENS = 100_000       # corpus size: shortest UltraDomain-mix documents until this many tokens
QUESTION_LIMIT = None         # e.g. 5 for a quick trial run; None = all questions
CHUNK_SIZE, CHUNK_OVERLAP = 1200, 100   # LightRAG defaults, used by both systems
NAIVE_TOP_K = 8               # 8 x 1200 tokens ~= 9.6k, inside the shared context budget
MAX_CONTEXT_TOKENS = 12_000   # retrieved-context budget for LightRAG

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path("/content/drive/MyDrive/COMP4330_LightRAG")
else:
    ROOT = Path("colab_local_test")
DATA = ROOT / "data"
RESULTS = ROOT / f"results_{BACKEND}"   # separate per backend: embedding sizes differ
for p in (DATA / "corpus", RESULTS):
    p.mkdir(parents=True, exist_ok=True)

if BACKEND == "openai":
    if IN_COLAB:
        from google.colab import userdata
        os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    BASE_URL, API_KEY = None, os.environ["OPENAI_API_KEY"]
    LLM_MODEL, EMBED_MODEL, EMBED_DIM = "gpt-4o-mini", "text-embedding-3-small", 1536
    LLM_EXTRA, MAX_ASYNC = {}, 4
else:
    BASE_URL, API_KEY = "http://localhost:11434/v1", "ollama"
    LLM_MODEL, EMBED_MODEL, EMBED_DIM = "qwen2.5-14b-16k", "bge-m3", 1024
    # qwen2.5:14b over-extracts entities and can loop until the context is full; cap the output.
    LLM_EXTRA, MAX_ASYNC = {"max_tokens": 3000}, 2
print("backend:", BACKEND, LLM_MODEL, EMBED_MODEL, "| saving to", ROOT)

## 2. Local model server (only for `BACKEND = "ollama"`)

In [ ]:
if BACKEND == "ollama":
    if IN_COLAB:
        subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True, check=True)
        subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        time.sleep(5)
    for m in ["qwen2.5:14b", "bge-m3"]:
        subprocess.run(["ollama", "pull", m], check=True, capture_output=True)
    # Ollama's OpenAI-compatible API cannot set the context size per request, so bake 16k into a model variant.
    Path("Modelfile").write_text("FROM qwen2.5:14b\nPARAMETER num_ctx 16384\nPARAMETER temperature 0\n")
    subprocess.run(["ollama", "create", LLM_MODEL, "-f", "Modelfile"], check=True, capture_output=True)
    print("ollama ready")

## 3. Data: UltraDomain `mix` subset
We keep the shortest documents until the total reaches `TARGET_TOKENS`, together with every question about them. Each question records its source document (`doc_id`), which serves as the ground truth for hit rate and nDCG.

In [ ]:
import tiktoken
from huggingface_hub import hf_hub_download

ENC = tiktoken.get_encoding("cl100k_base")
rows = [json.loads(l) for l in open(hf_hub_download("TommyChien/UltraDomain", "mix.jsonl", repo_type="dataset"), encoding="utf-8")]
doc_ids = {}
for r in rows:
    doc_ids.setdefault(r["context"], "doc_" + hashlib.md5(r["context"].encode()).hexdigest()[:10])

picked, total = set(), 0
for text in sorted(doc_ids, key=lambda t: len(ENC.encode(t))):
    n = len(ENC.encode(text))
    if total + n > TARGET_TOKENS:
        break
    picked.add(text); total += n

for text in picked:
    (DATA / "corpus" / f"{doc_ids[text]}.txt").write_text(text, encoding="utf-8")
QUESTIONS = [{"qid": f"q{i:03d}", "question": r["input"], "answers": r["answers"],
              "doc_id": doc_ids[r["context"]], "dataset": r["dataset"]}
             for i, r in enumerate(rows) if r["context"] in picked][:QUESTION_LIMIT]
DOCS = sorted((DATA / "corpus").glob("*.txt"))
DOCS = [p for p in DOCS if p.read_text(encoding="utf-8") in picked]
print(f"{len(DOCS)} documents, {total} tokens, {len(QUESTIONS)} questions")
print(collections.Counter(q["dataset"] for q in QUESTIONS))

## 4. Naive RAG baseline
Fixed-size chunks → embeddings → cosine top-k → one LLM call.

In [ ]:
from openai import OpenAI, AsyncOpenAI
client = OpenAI(base_url=BASE_URL, api_key=API_KEY)
aclient = AsyncOpenAI(base_url=BASE_URL, api_key=API_KEY)

NAIVE_PROMPT = """Answer the question using only the context below. If the context does not contain the answer, say so.
Reply in a single paragraph.

---Context---
{context}

---Question---
{question}"""

def chunk(text, size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    toks = ENC.encode(text)
    return [ENC.decode(toks[i:i + size]) for i in range(0, max(len(toks) - overlap, 1), size - overlap)]

def embed_sync(texts):
    vecs = np.array([d.embedding for d in client.embeddings.create(model=EMBED_MODEL, input=texts).data], dtype=np.float32)
    return vecs / np.linalg.norm(vecs, axis=1, keepdims=True)

def top_k(query_vec, matrix, k):
    return np.argsort(-(matrix @ query_vec))[:k]

NAIVE_DIR = RESULTS / "naive_index"
if (NAIVE_DIR / "vectors.npy").exists():
    CHUNKS = json.loads((NAIVE_DIR / "chunks.json").read_text(encoding="utf-8"))
    VECTORS = np.load(NAIVE_DIR / "vectors.npy")
else:
    CHUNKS = [{"doc_id": p.stem, "text": c} for p in DOCS for c in chunk(p.read_text(encoding="utf-8"))]
    VECTORS = np.concatenate([embed_sync([c["text"] for c in CHUNKS[i:i + 32]]) for i in range(0, len(CHUNKS), 32)])
    NAIVE_DIR.mkdir(exist_ok=True)
    np.save(NAIVE_DIR / "vectors.npy", VECTORS)
    (NAIVE_DIR / "chunks.json").write_text(json.dumps(CHUNKS, ensure_ascii=False), encoding="utf-8")

def naive_answer(question):
    t0 = time.perf_counter()
    retrieved = [CHUNKS[i] for i in top_k(embed_sync([question])[0], VECTORS, NAIVE_TOP_K)]
    context = "\n\n".join(f"[{n + 1}] {c['text']}" for n, c in enumerate(retrieved))
    r = client.chat.completions.create(model=LLM_MODEL, temperature=0, **LLM_EXTRA,
        messages=[{"role": "user", "content": NAIVE_PROMPT.format(context=context, question=question)}])
    return {"answer": r.choices[0].message.content,
            "retrieved_doc_ids": [c["doc_id"] for c in retrieved], "contexts": [c["text"] for c in retrieved],
            "prompt_tokens": r.usage.prompt_tokens, "completion_tokens": r.usage.completion_tokens,
            "llm_calls": 1, "latency_s": round(time.perf_counter() - t0, 2)}

# self-check
assert list(top_k(np.array([0.1, 0.9, 0.2]), np.eye(3), 2)) == [1, 2]
assert len(chunk("word " * 3000)) == 3
print(len(CHUNKS), "chunks indexed for naive RAG")

## 5. LightRAG index
The LLM extracts entities and relations from every chunk to build the knowledge graph. This is the slow and costly step. It resumes automatically after a disconnect.

In [ ]:
from lightrag import LightRAG, QueryParam
from lightrag.llm.openai import openai_complete_if_cache
from lightrag.utils import EmbeddingFunc, TokenTracker

TRACKER = TokenTracker()

async def llm(prompt, system_prompt=None, history_messages=None, **kw):
    return await openai_complete_if_cache(LLM_MODEL, prompt, system_prompt=system_prompt,
        history_messages=history_messages or [], base_url=BASE_URL, api_key=API_KEY,
        token_tracker=TRACKER, **{"temperature": 0, **LLM_EXTRA, **kw})

async def embed(texts, **_):
    r = await aclient.embeddings.create(model=EMBED_MODEL, input=texts)
    return np.array([d.embedding for d in r.data], dtype=np.float32)

rag = LightRAG(
    working_dir=str(RESULTS / "lightrag_index"),
    llm_model_func=llm, llm_model_name=LLM_MODEL, llm_model_max_async=MAX_ASYNC,
    embedding_func=EmbeddingFunc(embedding_dim=EMBED_DIM, max_token_size=8192, func=embed),
    chunk_token_size=CHUNK_SIZE, chunk_overlap_token_size=CHUNK_OVERLAP,
    enable_llm_cache=False,                     # measure query cost fresh every time
    enable_llm_cache_for_entity_extract=True,   # makes an interrupted index cheap to resume
)
await rag.initialize_storages()

TRACKER.reset(); t0 = time.perf_counter()
await rag.ainsert([p.read_text(encoding="utf-8") for p in DOCS], ids=[p.stem for p in DOCS],
                  file_paths=[p.name for p in DOCS])
index_cost = {"docs": len(DOCS), "index_time_s": round(time.perf_counter() - t0, 1), **TRACKER.get_usage()}
(RESULTS / "index_cost.json").write_text(json.dumps(index_cost, indent=2))
print("LightRAG index cost (this run only; 0 if already indexed):", index_cost)

## 6. Answer every question with both systems
LightRAG uses its default `mix` mode with reranking off. The results are written to `exp1_naive.jsonl` and `exp1_lightrag.jsonl`.

In [ ]:
async def lightrag_answer(question, mode="mix"):
    TRACKER.reset(); t0 = time.perf_counter()
    r = await rag.aquery_llm(question, param=QueryParam(mode=mode, response_type="Single Paragraph",
                                                        enable_rerank=False, max_total_tokens=MAX_CONTEXT_TOKENS))
    data, usage = r.get("data") or {}, TRACKER.get_usage()
    chunks = data.get("chunks", [])
    return {"answer": (r.get("llm_response") or {}).get("content", ""),
            "retrieved_doc_ids": [c.get("file_path", "").removesuffix(".txt") for c in chunks],
            "contexts": [c.get("content", "") for c in chunks],
            "n_entities": len(data.get("entities", [])), "n_relations": len(data.get("relationships", [])),
            "prompt_tokens": usage["prompt_tokens"], "completion_tokens": usage["completion_tokens"],
            "llm_calls": usage["call_count"], "latency_s": round(time.perf_counter() - t0, 2)}

def load_jsonl(path):
    return [json.loads(l) for l in open(path, encoding="utf-8")] if path.exists() else []

async def naive_async(q):
    return naive_answer(q)

for name, answer_fn in [("naive", naive_async), ("lightrag", lightrag_answer)]:
    out = RESULTS / f"exp1_{name}.jsonl"
    done = {r["qid"] for r in load_jsonl(out)}
    for i, q in enumerate(QUESTIONS):
        if q["qid"] in done:
            continue
        res = await answer_fn(q["question"])
        with out.open("a", encoding="utf-8") as f:
            f.write(json.dumps({**q, "system": name, **res}, ensure_ascii=False) + "\n")
        print(f"[{name}] {i + 1}/{len(QUESTIONS)} {q['qid']} {res['latency_s']}s")
print("done")

## 7. Evaluation
* **Retrieval: hit rate and nDCG.** These are measured at document level. A retrieved chunk counts as relevant when it comes from the question's source document.
* **Generation: RAGAS faithfulness and answer relevancy** (Es et al., 2024). The same LLM acts as the judge. This is a known bias, so it should be listed as a limitation.
* **Token F1** against the gold answer. This is a cheap sanity check.
* **Cost:** tokens, LLM calls and latency, copied from the run.

In [ ]:
from ragas.llms import llm_factory
from ragas.embeddings import embedding_factory
from ragas.metrics.collections import Faithfulness, AnswerRelevancy

def strip_references(answer):
    return re.split(r"\n#*\s*References\s*\n", answer)[0].strip()   # LightRAG appends a reference list

def norm_tokens(text):
    text = "".join(ch for ch in text.lower() if ch not in string.punctuation)
    return [w for w in text.split() if w not in {"a", "an", "the"}]

def token_f1(pred, golds):
    best = 0.0
    for gold in golds:
        common = sum((collections.Counter(norm_tokens(pred)) & collections.Counter(norm_tokens(gold))).values())
        if common:
            p, r = common / len(norm_tokens(pred)), common / len(norm_tokens(gold))
            best = max(best, 2 * p * r / (p + r))
    return best

def ndcg(retrieved, gold, n_gold_chunks):
    dcg = sum(1 / math.log2(i + 2) for i, d in enumerate(retrieved) if d == gold)
    ideal = sum(1 / math.log2(i + 2) for i in range(min(len(retrieved), n_gold_chunks)))
    return dcg / ideal if ideal else 0.0

assert strip_references("Ans.\n\n### References\n\n- [1] a.txt") == "Ans."
assert token_f1("The cat sat", ["cat sat down"]) == 0.8
assert ndcg(["a", "b"], "a", 1) == 1.0 and ndcg(["b", "a"], "a", 1) < 1 and ndcg(["b"], "a", 3) == 0

judge = llm_factory(LLM_MODEL, client=aclient, temperature=0, max_tokens=3000)
faith = Faithfulness(llm=judge)
relevancy = AnswerRelevancy(llm=judge, embeddings=embedding_factory("openai", EMBED_MODEL, client=aclient))
n_gold_chunks = collections.Counter(c["doc_id"] for c in CHUNKS)

SCORES = RESULTS / "scores.jsonl"
scored = {(s["system"], s["qid"]) for s in load_jsonl(SCORES)}
for name in ["naive", "lightrag"]:
    records = load_jsonl(RESULTS / f"exp1_{name}.jsonl")
    for i, rec in enumerate(records):
        if (name, rec["qid"]) in scored:
            continue
        answer = strip_references(rec["answer"])
        s = {"system": name, "qid": rec["qid"],
             "hit": float(rec["doc_id"] in rec["retrieved_doc_ids"]),
             "ndcg": ndcg(rec["retrieved_doc_ids"], rec["doc_id"], n_gold_chunks[rec["doc_id"]]),
             "token_f1": token_f1(answer, rec["answers"]),
             **{k: rec[k] for k in ("prompt_tokens", "completion_tokens", "llm_calls", "latency_s")}}
        for metric, coro in [("faithfulness", faith.ascore(user_input=rec["question"], response=answer,
                                                         retrieved_contexts=rec["contexts"] or [""])),
                             ("answer_relevancy", relevancy.ascore(user_input=rec["question"], response=answer))]:
            try:
                s[metric] = (await coro).value
            except Exception as e:   # one malformed judge reply loses one score, not the run
                print(f"  {metric} failed for {name}/{rec['qid']}: {str(e)[:120]}")
                s[metric] = None
        with SCORES.open("a", encoding="utf-8") as f:
            f.write(json.dumps(s) + "\n")
        print(f"[{name}] {i + 1}/{len(records)} faith={s['faithfulness']} rel={s['answer_relevancy']} hit={s['hit']}")
print("scoring done")

## 8. Results table (copy into slides and the report)

In [ ]:
import pandas as pd
df = pd.DataFrame(load_jsonl(SCORES))
summary = df.groupby("system")[["hit", "ndcg", "faithfulness", "answer_relevancy", "token_f1",
                                "prompt_tokens", "completion_tokens", "llm_calls", "latency_s"]].mean().round(3)
summary.insert(0, "n", df.groupby("system").size())
summary.to_csv(RESULTS / "exp1_summary.csv")
print("LightRAG index cost:", json.loads((RESULTS / "index_cost.json").read_text()))
summary